# 02 — MSP-Podcast→HCUDB 4クラスdecoder学習・評価

検証済みframe cacheとmanifestだけを入力にし、MSP-Podcast学習、HCUDB継続学習、両データセットの追加学習前後評価を行います。IEMOCAPは今回の一括研究経路には含めません。実データ1 epoch疎通と正式実行は別の出力先を使い、どちらも既定で無効です。


In [13]:
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ser_pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ser_pipeline.notebook_api import environment_summary
from ser_pipeline.study import (
    DatasetArtifacts, prepare_study_stores, require_formal_epochs,
    run_transfer_study, summarize_study,
)
from ser_pipeline.training import TrainingConfig

STUDY_DATASETS = ("msp_podcast", "hcudb1")

RUN_REAL_SMOKE = False
RUN_FORMAL_SEED_42 = False

FORMAL_EPOCHS = 10
CONFIRM_CACHE_VALIDATION = False
CONFIRM_BENCHMARK_AND_CAPACITY = False
CONFIRM_SMOKE_COMPLETED = True
CONFIRM_SEED_42_ARTIFACTS = True

RUN_FORMAL_SEEDS_43_44 = True

ARTIFACT_DIR = PROJECT_ROOT / 'runs' / 'ser_decoder_study'


def load_study_artifacts():
    artifacts = {}
    missing = []
    for name in STUDY_DATASETS:
        manifest_key = f'SER_{name.upper()}_MANIFEST'
        cache_key = f'SER_{name.upper()}_CACHE'
        if not os.environ.get(manifest_key):
            missing.append(manifest_key)
        if not os.environ.get(cache_key):
            missing.append(cache_key)
        exclusion_contract_path = None
        duplicate_audit_path = None
        duplicate_exclusion_contract_path = None
        if name == 'msp_podcast':
            exclusion_key = 'SER_MSP_PODCAST_EXCLUSION_CONTRACT'
            duplicate_audit_key = 'SER_MSP_PODCAST_DUPLICATE_AUDIT'
            duplicate_exclusion_key = 'SER_MSP_PODCAST_DUPLICATE_EXCLUSION_CONTRACT'
            for provenance_key in (exclusion_key, duplicate_audit_key, duplicate_exclusion_key):
                if not os.environ.get(provenance_key):
                    missing.append(provenance_key)
            if exclusion_key not in missing:
                exclusion_contract_path = Path(os.environ[exclusion_key])
            if duplicate_audit_key not in missing:
                duplicate_audit_path = Path(os.environ[duplicate_audit_key])
            if duplicate_exclusion_key not in missing:
                duplicate_exclusion_contract_path = Path(os.environ[duplicate_exclusion_key])
        if manifest_key not in missing and cache_key not in missing:
            artifacts[name] = DatasetArtifacts(
                manifest_path=Path(os.environ[manifest_key]),
                cache_root=Path(os.environ[cache_key]),
                exclusion_contract_path=exclusion_contract_path,
                duplicate_audit_path=duplicate_audit_path,
                duplicate_exclusion_contract_path=duplicate_exclusion_contract_path,
            )
    if missing:
        raise ValueError(f'Missing study artifact environment variables: {missing}')
    return artifacts


def validate_execution_gates():
    if not CONFIRM_CACHE_VALIDATION:
        raise RuntimeError('Confirm both MSP/HCUDB caches were completely validated')
    if not CONFIRM_BENCHMARK_AND_CAPACITY:
        raise RuntimeError('Confirm the one-item CPU benchmark and the +20% capacity gate')
    artifacts = load_study_artifacts()
    stores = prepare_study_stores(artifacts)
    cache_validation = {name: store.validation_report for name, store in stores.items()}
    return artifacts, cache_validation, stores


## 1. cache-only実行環境


In [15]:
environment_summary()


{'python': '3.10.20',
 'platform': 'Linux-6.6.114.1-microsoft-standard-WSL2-x86_64-with-glibc2.35',
 'pytorch': '2.12.1+cu130',
 'cuda_available': False,
 'label_order': ['anger', 'happy', 'sadness', 'disgust'],
 'feature_layer': 'final_after_encoder_norm'}

## 2. 実行設定の確認


In [16]:
{
    'datasets': STUDY_DATASETS,
    'device': 'cpu',
    'run_real_smoke': RUN_REAL_SMOKE,
    'run_formal_seed_42': RUN_FORMAL_SEED_42,
    'run_formal_seeds_43_44': RUN_FORMAL_SEEDS_43_44,
    'formal_epochs': FORMAL_EPOCHS,
}


{'datasets': ('msp_podcast', 'hcudb1'),
 'device': 'cpu',
 'run_real_smoke': False,
 'run_formal_seed_42': False,
 'run_formal_seeds_43_44': True,
 'formal_epochs': 10}

## 3. 実データ1 epoch疎通（正式集計外）

seed 42でMSP親学習→HCUDB継続学習→両データセットの前後評価を行います。出力は`smoke/`に隔離され、正式結果には混ぜません。


In [5]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "ser_pipeline").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

MANIFEST_DIR = PROJECT_ROOT / "runs" / "ser_manifests"
CACHE_DIR = PROJECT_ROOT / "runs" / "ser_feature_cache"

artifact_paths = {
    "SER_MSP_PODCAST_MANIFEST":
        MANIFEST_DIR / "msp_podcast_4class_v1.jsonl",

    "SER_MSP_PODCAST_CACHE":
        CACHE_DIR / "msp_podcast_base_final_v1",

    "SER_MSP_PODCAST_EXCLUSION_CONTRACT":
        MANIFEST_DIR / "msp_missing_audio_exclusions_v1.json",

    "SER_MSP_PODCAST_DUPLICATE_AUDIT":
        MANIFEST_DIR / "msp_audio_duplicate_audit_v1.json",

    "SER_MSP_PODCAST_DUPLICATE_EXCLUSION_CONTRACT":
        MANIFEST_DIR / "msp_audio_duplicate_exclusions_v1.json",

    "SER_HCUDB1_MANIFEST":
        MANIFEST_DIR / "hcudb1_4class_v1.jsonl",

    "SER_HCUDB1_CACHE":
        CACHE_DIR / "hcudb1_base_final_v1",
}

for name, path in artifact_paths.items():
    os.environ[name] = str(path)

{
    name: {
        "path": str(path),
        "exists": path.exists(),
    }
    for name, path in artifact_paths.items()
}

{'SER_MSP_PODCAST_MANIFEST': {'path': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_manifests/msp_podcast_4class_v1.jsonl',
  'exists': True},
 'SER_MSP_PODCAST_CACHE': {'path': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_feature_cache/msp_podcast_base_final_v1',
  'exists': True},
 'SER_MSP_PODCAST_EXCLUSION_CONTRACT': {'path': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_manifests/msp_missing_audio_exclusions_v1.json',
  'exists': True},
 'SER_MSP_PODCAST_DUPLICATE_AUDIT': {'path': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_manifests/msp_audio_duplicate_audit_v1.json',
  'exists': True},
 'SER_MSP_PODCAST_DUPLICATE_EXCLUSION_CONTRACT': {'path': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_manifests/msp_audio_duplicate_exclusions_v1.json',
  'exists': True},
 'SER_HCUDB1_MANIFEST': {'path': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_manifests/hcudb1_4class_v1.jsonl',
  'exists': True},
 'SER_HCUDB1_CACHE': {'path': '/mnt/

In [6]:
if RUN_REAL_SMOKE:
    smoke_artifacts, smoke_cache_validation, smoke_stores = validate_execution_gates()
    smoke_summary = run_transfer_study(
        smoke_artifacts,
        ARTIFACT_DIR / 'smoke',
        seeds=(42,),
        base_config=TrainingConfig(seed=42, device='cpu', epochs=1),
        stores=smoke_stores,
    )
else:
    smoke_summary = {'status': 'disabled_by_default', 'seed': 42, 'epochs': 1}
summarize_study(smoke_summary)


{'seeds': [42],
 'runs': [{'seed': 42,
           'parent': {'best_epoch': 1,
                      'validation': {'uar': 0.46601907827876704,
                                     'macro_f1': 0.45142850990410816,
                                     'wa': 0.6633333333333333,
                                     'loss': 0.872211251915573},
                      'seconds': None,
                      'checkpoint': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_decoder_study/smoke/seed-42/checkpoints/msp/msp_train_seed42_best.pt'},
           'child': {'best_epoch': 1,
                     'validation': {'uar': 0.44583333333333336,
                                    'macro_f1': 0.4477644011649881,
                                    'wa': 0.5333333333333333,
                                    'loss': 1.1009193660538346},
                     'seconds': None,
                     'checkpoint': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_decoder_study/smoke/seed-42/chec

In [7]:
run = smoke_summary["runs"][0]
metric_names = ("loss", "accuracy", "wa", "uar", "macro_f1")

def show_metrics(label, metrics):
    values = {
        name: round(float(metrics[name]), 4)
        for name in metric_names
        if name in metrics and metrics[name] is not None
    }
    print(label, values)

show_metrics(
    "MSP validation:",
    run["parent"]["best_validation_metrics"],
)
show_metrics(
    "HCUDB validation:",
    run["child"]["best_validation_metrics"],
)

for dataset in ("msp_podcast", "hcudb1"):
    show_metrics(
        f"{dataset} before:",
        run["before"][dataset]["result"],
    )
    show_metrics(
        f"{dataset} after:",
        run["after"][dataset]["result"],
    )

print("MSP history:", run["parent"]["history"])
print("HCUDB history:", run["child"]["history"])

MSP validation: {'loss': 0.8722, 'accuracy': 0.6633, 'wa': 0.6633, 'uar': 0.466, 'macro_f1': 0.4514}
HCUDB validation: {'loss': 1.1009, 'accuracy': 0.5333, 'wa': 0.5333, 'uar': 0.4458, 'macro_f1': 0.4478}
msp_podcast before: {}
msp_podcast after: {}
hcudb1 before: {}
hcudb1 after: {}
MSP history: [{'epoch': 1, 'train_loss': 0.8736875104549495, 'validation': {'accuracy': 0.6633333333333333, 'wa': 0.6633333333333333, 'uar': 0.46601907827876704, 'macro_f1': 0.45142850990410816, 'loss': 0.872211251915573, 'reported_class_indices': [0, 1, 2, 3], 'class_metrics': [{'class_index': 0, 'class_label': 'anger', 'precision': 0.6386182462356067, 'recall': 0.6906130268199234, 'f1': 0.6635987114588127, 'support': 1044}, {'class_index': 1, 'class_label': 'happy', 'precision': 0.6896853146853147, 'recall': 0.8727876106194691, 'f1': 0.7705078125, 'support': 1808}, {'class_index': 2, 'class_label': 'sadness', 'precision': 0.48633879781420764, 'recall': 0.30067567567567566, 'f1': 0.37160751565762, 'suppor

## 4. 正式seed 42実行ゲート

1 epoch疎通の時間と履歴を確認して`FORMAL_EPOCHS`を正の整数に固定し、先にseed 42だけを実行します。未設定のまま実行すると拒否します。


In [9]:
formal_output = ARTIFACT_DIR / "formal" / "initial-seed-42"

print("output:", formal_output)
print("already exists:", formal_output.exists())

output: /mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_decoder_study/formal/initial-seed-42
already exists: False


In [12]:
if RUN_FORMAL_SEED_42:
    if not CONFIRM_SMOKE_COMPLETED:
        raise RuntimeError('Confirm the real-data 1 epoch smoke run before formal seed 42')
    formal_epochs = require_formal_epochs(FORMAL_EPOCHS)
    formal_artifacts, formal_cache_validation, formal_stores = validate_execution_gates()
    formal_seed_42_summary = run_transfer_study(
        formal_artifacts,
        ARTIFACT_DIR / 'formal' / 'initial-seed-42',
        seeds=(42,),
        base_config=TrainingConfig(seed=42, device='cpu', epochs=formal_epochs),
        stores=formal_stores,
    )
else:
    formal_seed_42_summary = {
        'status': 'disabled_by_default', 'seed': 42, 'formal_epochs': FORMAL_EPOCHS
    }
summarize_study(formal_seed_42_summary)


{'seeds': [42],
 'runs': [{'seed': 42,
           'parent': {'best_epoch': 6,
                      'validation': {'uar': 0.5272351616187442,
                                     'macro_f1': 0.5221168474984074,
                                     'wa': 0.6302777777777778,
                                     'loss': 0.9615271460824942},
                      'seconds': None,
                      'checkpoint': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_decoder_study/formal/initial-seed-42/seed-42/checkpoints/msp/msp_train_seed42_best.pt'},
           'child': {'best_epoch': 4,
                     'validation': {'uar': 0.5458333333333334,
                                    'macro_f1': 0.5408799549374108,
                                    'wa': 0.5666666666666667,
                                    'loss': 1.1181496739195818},
                     'seconds': None,
                     'checkpoint': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_decoder_study/for

In [13]:
run = formal_seed_42_summary["runs"][0]

def compact(metrics):
    return {
        key: round(float(metrics[key]), 4)
        for key in ("loss", "accuracy", "wa", "uar", "macro_f1")
        if metrics.get(key) is not None
    }

print("MSP best epoch:", run["parent"]["best_epoch"])
print("MSP validation:", compact(run["parent"]["best_validation_metrics"]))

print("\nHCUDB best epoch:", run["child"]["best_epoch"])
print("HCUDB validation:", compact(run["child"]["best_validation_metrics"]))

for dataset in ("msp_podcast", "hcudb1"):
    print(f"\n{dataset}")
    for stage in ("before", "after"):
        metrics = run[stage][dataset]["result"]["metrics_4class"]
        print(stage, compact(metrics))

print("\nMSP validation history")
for row in run["parent"]["history"]:
    validation = row["validation"]
    print(
        {
            "epoch": row["epoch"],
            "train_loss": round(row["train_loss"], 4),
            "val_loss": round(validation["loss"], 4),
            "val_uar": round(validation["uar"], 4),
            "val_macro_f1": round(validation["macro_f1"], 4),
        }
    )

print("\nHCUDB validation history")
for row in run["child"]["history"]:
    validation = row["validation"]
    print(
        {
            "epoch": row["epoch"],
            "train_loss": round(row["train_loss"], 4),
            "val_loss": round(validation["loss"], 4),
            "val_uar": round(validation["uar"], 4),
            "val_macro_f1": round(validation["macro_f1"], 4),
        }
    )

print("\nMSP best class metrics")
for row in run["parent"]["best_validation_metrics"]["class_metrics"]:
    print(row)

print("\nHCUDB best class metrics")
for row in run["child"]["best_validation_metrics"]["class_metrics"]:
    print(row)

MSP best epoch: 6
MSP validation: {'loss': 0.9615, 'accuracy': 0.6303, 'wa': 0.6303, 'uar': 0.5272, 'macro_f1': 0.5221}

HCUDB best epoch: 4
HCUDB validation: {'loss': 1.1181, 'accuracy': 0.5667, 'wa': 0.5667, 'uar': 0.5458, 'macro_f1': 0.5409}

msp_podcast
before {'loss': 0.8425, 'accuracy': 0.6836, 'wa': 0.6836, 'uar': 0.4977, 'macro_f1': 0.5012}
after {'loss': 1.214, 'accuracy': 0.5307, 'wa': 0.5307, 'uar': 0.494, 'macro_f1': 0.4366}

hcudb1
before {'loss': 2.1277, 'accuracy': 0.3433, 'wa': 0.3433, 'uar': 0.2667, 'macro_f1': 0.2347}
after {'loss': 1.1837, 'accuracy': 0.56, 'wa': 0.56, 'uar': 0.5125, 'macro_f1': 0.5042}

MSP validation history
{'epoch': 1, 'train_loss': 0.8737, 'val_loss': 0.8722, 'val_uar': 0.466, 'val_macro_f1': 0.4514}
{'epoch': 2, 'train_loss': 0.7589, 'val_loss': 0.8658, 'val_uar': 0.5023, 'val_macro_f1': 0.4886}
{'epoch': 3, 'train_loss': 0.6918, 'val_loss': 0.8938, 'val_uar': 0.4939, 'val_macro_f1': 0.4766}
{'epoch': 4, 'train_loss': 0.6363, 'val_loss': 0.9484

## 5. 正式seed 43・44実行ゲート

seed 42のcheckpoint、評価signature、cache ID、設定値を確認した後だけ実行します。出力はseed 42の正式出力と分けて保存します。


In [17]:
if RUN_FORMAL_SEEDS_43_44:
    if not CONFIRM_SEED_42_ARTIFACTS:
        raise RuntimeError('Confirm the formal seed 42 artifacts before seeds 43 and 44')
    formal_epochs = require_formal_epochs(FORMAL_EPOCHS)
    followup_artifacts, followup_cache_validation, followup_stores = validate_execution_gates()
    formal_followup_summary = run_transfer_study(
        followup_artifacts,
        ARTIFACT_DIR / 'formal' / 'followup-seeds-43-44',
        seeds=(43, 44),
        base_config=TrainingConfig(seed=43, device='cpu', epochs=formal_epochs),
        stores=followup_stores,
    )
else:
    formal_followup_summary = {
        'status': 'disabled_by_default', 'seeds': [43, 44], 'formal_epochs': FORMAL_EPOCHS
    }
summarize_study(formal_followup_summary)


{'seeds': [43, 44],
 'runs': [{'seed': 43,
           'parent': {'best_epoch': 9,
                      'validation': {'uar': 0.5354925894005083,
                                     'macro_f1': 0.5003594948571375,
                                     'wa': 0.5852777777777778,
                                     'loss': 1.224340573058555},
                      'seconds': None,
                      'checkpoint': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_decoder_study/formal/followup-seeds-43-44/seed-43/checkpoints/msp/msp_train_seed43_best.pt'},
           'child': {'best_epoch': 10,
                     'validation': {'uar': 0.55,
                                    'macro_f1': 0.5259718576990631,
                                    'wa': 0.57,
                                    'loss': 1.4208979459119973},
                     'seconds': None,
                     'checkpoint': '/mnt/c/Users/RD004/Documents/lab/emotion2vec/runs/ser_decoder_study/formal/followup-seeds-

## 6. MSP単体：クラス重み付き損失の比較

更新後はカーネルを再起動し、この節の **6.1 → 6.2 → 6.3 → 6.4** だけを実行します。
この節は独立しており、上のsetupや「3〜5」の学習セルを実行する必要はありません。
MSPの重みなし結果を読み込み、同じseedの初期値から重みありモデルを10 epoch学習します。
バッチサイズ8、学習率0.001、Dropout 0、データ分割と発話順序を維持します。
重みは **trainの総件数 / (4 × trainのクラス別件数)**。validationのUARを主指標に比較します。
validationのlossは従来の重みなし計算です。重みあり/なしのtrain lossは同じ尺度として比較しません。
HCUDBの学習とtest評価はこの節では実行しません。


### 6.1 比較設定


In [14]:
import json, os, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'ser_pipeline').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from ser_pipeline.cache import ShardedFeatureStore
from ser_pipeline.study import DatasetArtifacts, load_msp_comparison_baselines, run_msp_loss_comparison
from ser_pipeline.training import TrainingConfig, training_loss_config

RUN_MSP_WEIGHTED_TRAINING = True  # 学習を開始するとき True にする
MSP_COMPARISON_SEEDS = (42,)  # 次に (43, 44) で同じ比較を行う
MSP_COMPARISON_CONFIG = TrainingConfig(device='cpu', epochs=10, batch_size=8)
MSP_COMPARISON_OUTPUT = PROJECT_ROOT / 'runs' / 'msp_class_weight_comparison' / (
    'seeds-' + '-'.join(str(seed) for seed in MSP_COMPARISON_SEEDS)
)

manifest_dir = PROJECT_ROOT / 'runs' / 'ser_manifests'
def msp_input(env_key, default):
    return Path(os.environ.get(env_key) or default)

MSP_COMPARISON_ARTIFACT = DatasetArtifacts(
    manifest_path=msp_input('SER_MSP_PODCAST_MANIFEST', manifest_dir / 'msp_podcast_4class_v1.jsonl'),
    cache_root=msp_input('SER_MSP_PODCAST_CACHE', PROJECT_ROOT / 'runs' / 'ser_feature_cache' / 'msp_podcast_base_final_v1'),
    exclusion_contract_path=msp_input('SER_MSP_PODCAST_EXCLUSION_CONTRACT', manifest_dir / 'msp_missing_audio_exclusions_v1.json'),
    duplicate_audit_path=msp_input('SER_MSP_PODCAST_DUPLICATE_AUDIT', manifest_dir / 'msp_audio_duplicate_audit_v1.json'),
    duplicate_exclusion_contract_path=msp_input('SER_MSP_PODCAST_DUPLICATE_EXCLUSION_CONTRACT', manifest_dir / 'msp_audio_duplicate_exclusions_v1.json'),
)
MSP_BASELINE_SUMMARIES = [
    PROJECT_ROOT / 'runs' / 'ser_decoder_timing_check_20260903' / 'formal' / 'initial-seed-42' / 'study_summary.json',
    PROJECT_ROOT / 'runs' / 'ser_decoder_study' / 'formal' / 'followup-seeds-43-44' / 'study_summary.json',
]
print('比較seed:', MSP_COMPARISON_SEEDS, '学習を実行:', RUN_MSP_WEIGHTED_TRAINING)
print('保存先:', MSP_COMPARISON_OUTPUT)


ImportError: cannot import name 'load_msp_comparison_baselines' from 'ser_pipeline.study' (/mnt/c/Users/RD004/Documents/lab/emotion2vec/ser_pipeline/study.py)

### 6.2 キャッシュ・比較元・クラス重みの確認

初回のキャッシュ完全検証には数分かかります。学習済み比較元と設定・manifest・cache・checkpointを照合します。
同じカーネルでこのセルを再実行した場合は、入力に変更がなければ検証済みstoreを再利用します。


In [ ]:
if MSP_COMPARISON_OUTPUT.exists() and any(MSP_COMPARISON_OUTPUT.iterdir()):
    raise ValueError('保存先に結果があります。6.4で確認するか、6.1で別の保存先を設定してください。')
if 'msp_comparison_store' not in globals():
    print('MSPキャッシュの完全検証を開始します。', flush=True)
    msp_comparison_store = ShardedFeatureStore(
        MSP_COMPARISON_ARTIFACT.cache_root, MSP_COMPARISON_ARTIFACT.manifest_path,
    )
else:
    msp_comparison_store.require_paths(MSP_COMPARISON_ARTIFACT.cache_root, MSP_COMPARISON_ARTIFACT.manifest_path)
    msp_comparison_store.ensure_validated()
msp_comparison_baselines = load_msp_comparison_baselines(
    MSP_BASELINE_SUMMARIES, msp_comparison_store, MSP_COMPARISON_CONFIG, MSP_COMPARISON_SEEDS,
)
msp_comparison_loss = training_loss_config(msp_comparison_store, 'msp_podcast', 'balanced')
display(pd.DataFrame({
    '感情': msp_comparison_loss['label_order'],
    'train件数': msp_comparison_loss['train_class_counts'],
    '重み': msp_comparison_loss['class_weights'],
}).round(4))
print('比較元の照合完了。seed:', list(msp_comparison_baselines))


### 6.3 MSPの重みあり学習を実行


In [ ]:
if RUN_MSP_WEIGHTED_TRAINING:
    msp_loss_comparison = run_msp_loss_comparison(
        MSP_COMPARISON_ARTIFACT, MSP_COMPARISON_OUTPUT, MSP_BASELINE_SUMMARIES,
        seeds=MSP_COMPARISON_SEEDS, base_config=MSP_COMPARISON_CONFIG, store=msp_comparison_store,
    )
else:
    print('学習は無効です。6.1でRUN_MSP_WEIGHTED_TRAININGをTrueにして実行してください。')


### 6.4 validation結果を比較

`none`は保存済みの重みなし結果、`balanced`は今回の重みあり結果です。
差分は **重みあり − 重みなし**。UAR・macro F1・WA・再現率は大きいほど良く、lossは小さいほど良い指標です。
seed 42で動作を確認したら、6.1のseedを`(43, 44)`に変えて比較し、3 seedでの傾向を確認します。


In [ ]:
msp_comparison_summary_path = MSP_COMPARISON_OUTPUT / 'comparison_summary.json'
if msp_comparison_summary_path.is_file():
    msp_loss_comparison = json.loads(msp_comparison_summary_path.read_text(encoding='utf-8'))
    print('validation結果:')
    display(pd.DataFrame(msp_loss_comparison['rows']).round(4))
    print('validation差分（重みあり − 重みなし）:')
    display(pd.DataFrame([
        {'seed': run['seed'], **run['validation_deltas']} for run in msp_loss_comparison['runs']
    ]).round(4))
    print('完了seed:', msp_loss_comparison['completed_seeds'], '/', msp_loss_comparison['requested_seeds'])
    print('比較実行時間（分）:', round(msp_loss_comparison['seconds'] / 60, 2))
    print('保存先:', msp_comparison_summary_path)
else:
    print('比較結果はまだありません。6.3の学習を完了してください。')
